# Language Translation System for Multilingual Customer Support
### Lexora AI Pvt. Ltd. — Technical Assessment (Ref: LX-NLP-2026-014)
**Role:** Natural Language Processing Engineer · **Domain:** Multilingual Ticket & Chat Translation

This notebook implements an end-to-end, **Transformer encoder–decoder** translation system that
translates customer-support messages between a customer's language and the support agent's language
while preserving **intent, tone, and technical accuracy**.

**Model used throughout:** `facebook/nllb-200-distilled-600M` (No Language Left Behind, 600M distilled).

**Supported language pairs (bidirectional with English):**
English ↔ French · English ↔ Spanish · English ↔ Hindi · English ↔ Tamil

> The notebook is designed to run top-to-bottom on a **fresh Google Colab runtime** (GPU recommended:
> *Runtime → Change runtime type → T4 GPU*). Every cell is complete and runnable — no placeholders.

---
### Notebook map
1. Environment Setup 2. Business Understanding · Model Selection 3. Data Collection
4. Data Understanding 5. Data Preprocessing 6. Language Detection
7. Transformer Architecture Explanation 8. Load Translation Model 9. Translation Pipeline
10. Post-Processing 11. Evaluation (BLEU / chrF / COMET) 12. Interactive Demo 13. Dry Run


## 1 · Environment Setup
We install a small, mutually-compatible set of libraries and **do not reinstall PyTorch**, so we keep
Colab's pre-built CUDA wheel and avoid driver/CUDA mismatches.

| Package | Purpose |
|---|---|
| `transformers`, `sentencepiece` | NLLB model + SentencePiece tokenizer |
| `datasets` | Load the OPUS-100 multilingual parallel corpus |
| `sacrebleu` | BLEU and chrF metrics (reference implementation) |
| `unbabel-comet` | COMET learned metric |
| `lingua-language-detector` | Robust language detection (works well on short text) |
| `matplotlib` | Visualisations |

We pin only where it matters and let `pip` resolve the rest against Colab's existing PyTorch.

In [ ]:
# --- 1.1 Install (quiet). Runs on a fresh Colab runtime. ---
# We do NOT install torch: Colab already ships a CUDA-matched build.
!pip install -q -U "transformers>=4.41,<5" sentencepiece "datasets>=2.19" \
    "sacrebleu>=2.4" "lingua-language-detector>=2.0" "unbabel-comet>=2.2" 2>/dev/null
print("Install step finished.")

In [ ]:
# --- 1.2 Verify installation & report environment ---
import platform
import torch, transformers, datasets, sacrebleu

print("Python           :", platform.python_version())
print("PyTorch          :", torch.__version__)
print("Transformers     :", transformers.__version__)
print("Datasets         :", datasets.__version__)
print("sacrebleu        :", sacrebleu.__version__)
try:
    import lingua; print("lingua           : installed")
except Exception as e:
    print("lingua           : NOT installed ->", e)
try:
    import comet; print("unbabel-comet    : installed")
except Exception as e:
    print("unbabel-comet    : NOT installed ->", e)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("\nCUDA available   :", torch.cuda.is_available())
print("Active device    :", DEVICE)
if DEVICE == "cuda":
    print("GPU              :", torch.cuda.get_device_name(0))

In [ ]:
# --- 1.3 Reproducibility: set all random seeds ---
import random, numpy as np, torch
SEED = 42
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()
print(f"Random seeds set to {SEED}.")

## 2 · Business Understanding

**Assessment objective.** Build a system that, given a support message in any supported source
language (possibly informal, containing emoji/typos, technical vocabulary such as error codes and
product names, or mixed-language content), produces a fluent, faithful translation in a specified
target language that a support agent or customer can use *without further manual editing*.

**The business problem.** Global support platforms receive tickets, live-chat messages, and emails in
dozens of languages, but agents are staffed in only a few. Manual translation is slow, expensive, and
inconsistent, which inflates first-response time and hurts CSAT. An automated, high-quality translation
layer lets any agent serve any customer in near real time.

**Why multilingual translation matters in customer support specifically.**
- **Coverage without headcount:** one agent pool can serve many locales, including low-resource Indian
  languages (Hindi, Tamil) that are hard to staff for.
- **Speed & cost:** automatic translation removes a manual step from every non-English ticket, cutting
  handle time and cost-per-contact.
- **Fidelity is safety-critical:** an urgent complaint must stay urgent, and identifiers such as error
  codes, order numbers, and product names must **survive translation unchanged** — a mistranslated
  error code sends the customer to the wrong fix. This drives our post-processing design (Section 10).
- **Real-world noise:** support text is messy (abbreviations, emoji, typos, code-switching), so the
  system must be *robust*, not just accurate on clean text (Section 6 & 10).

### 2.1 · Model Selection & Justification

We evaluated four pretrained Transformer encoder–decoder models before committing to one.

| Model | Params | Languages | Indian-lang support | Notes for this task |
|---|---|---|---|---|
| **NLLB-200 Distilled 600M** (`facebook/nllb-200-distilled-600M`) | 600M | **200** | **Excellent** (Hindi, Tamil, + many more) | Purpose-built many-to-many MT; single model covers all our pairs; distilled → fits Colab. |
| **MarianMT** (`Helsinki-NLP/opus-mt-*`) | ~75M each | Bilingual per checkpoint | Limited / uneven for Indic | Great quality but needs a **separate model per direction** → heavy to manage 8 directions. |
| **mBART-50** (`facebook/mbart-large-50-many-to-many-mmt`) | 610M | 50 | Good (incl. hi; **no Tamil**) | Strong MT model but **Tamil is not in the 50** → fails a required pair. |
| **mT5** (`google/mt5-*`) | 300M–1.3B+ | 101 | Present but **needs task fine-tuning** | Pretrained on span-corruption, *not* translation-ready out of the box → extra training. |

**Why NLLB-200 Distilled 600M is selected** (used consistently for the rest of the notebook):
- **Multilingual support:** one many-to-many model spanning 200 languages covers *all four* required
  pairs — no per-pair model juggling (unlike MarianMT).
- **Indian language support:** first-class Hindi (`hin_Deva`) **and** Tamil (`tam_Taml`), which mBART-50
  lacks — directly satisfies the low/medium-resource Indian-language requirement.
- **Encoder–decoder architecture:** a true seq2seq Transformer (the architecture the assessment
  mandates), so self-attention, cross-attention, and multi-head attention all apply cleanly (Section 7).
- **Transfer learning:** a large pretrained multilingual checkpoint we reuse directly (zero-shot for our
  pairs) and could fine-tune — exactly the "pretrained encoder–decoder leveraged via transfer learning"
  the brief calls for.
- **Robustness:** trained on massive, noisy, web-scale multilingual data, so it handles informal text,
  code-switching, and typos better than a narrowly-trained bilingual model.
- **Context preservation:** attention over the full sequence preserves intent/tone; combined with our
  masking of technical identifiers (Section 10), it keeps meaning and error codes intact.
- **Production quality:** the **distilled 600M** variant balances quality against latency/memory, so it
  runs on a single Colab GPU while staying deployable — unlike the 1.3B/3.3B NLLB or large mT5.

> Decision is final for this notebook: **we use `facebook/nllb-200-distilled-600M` everywhere.**

## 3 · Data Collection

**Dataset chosen: OPUS-100** (`Helsinki-NLP/opus-100`).

**Why it satisfies the assessment:**
- It is an English-centric multilingual corpus with **100 language pairs**, including **all four** we
  need: `en-fr`, `en-es`, `en-hi`, `en-ta`. This gives us both high-resource (fr, es) and
  low/medium-resource **Indian** pairs (hi, ta) in one consistent source.
- Each config ships with **train / validation / test** splits already, so we can honour the standard
  ML split discipline and evaluate on held-out references (needed for BLEU/chrF/COMET in Section 11).
- It is drawn from diverse OPUS sources (subtitles, web, docs), i.e. varied styles — supporting the
  **generalisation** requirement rather than a single narrow domain.

We use it for **data understanding** and as a source of **held-out reference pairs for evaluation**.
NLLB is used zero-shot (no fine-tuning required), so we do not need the train split for training; we
keep it only to describe the split and to draw naturalistic sentences.

In [ ]:
# --- 3.1 Load OPUS-100 for each required pair ---
from datasets import load_dataset

PAIRS = ["en-fr", "en-es", "en-hi", "en-ta"]   # all four required pairs
raw = {}
for pair in PAIRS:
    raw[pair] = load_dataset("Helsinki-NLP/opus-100", pair)
    print(f"Loaded opus-100 [{pair}]:",
          {split: raw[pair][split].num_rows for split in raw[pair]})

In [ ]:
# --- 3.2 Display a few parallel samples per pair ---
def show_samples(pair, n=3, split="test"):
    print(f"\n=== {pair}  ({split}) ===")
    for ex in raw[pair][split].select(range(n)):
        src, tgt = pair.split("-")
        print(f"  {src}: {ex['translation'][src]}")
        print(f"  {tgt}: {ex['translation'][tgt]}")
        print("  " + "-" * 60)

for p in PAIRS:
    show_samples(p, n=3)

**Train / validation / test split.** OPUS-100 provides these splits per pair out of the box:
- **train** — largest split; used to *learn* parameters if one fine-tunes. We rely on NLLB's own
  pretraining (transfer learning), so we do **not** train on it here.
- **validation** — used for hyper-parameter/checkpoint selection during fine-tuning.
- **test** — a held-out set with human references, which we use in Section 11 to compute BLEU, chrF,
  and COMET. Keeping test strictly unseen is what makes the quality numbers meaningful and supports the
  **generalisation** criterion.

## 4 · Data Understanding
We inspect corpus size, the languages/scripts involved, and sentence-length distributions, then
visualise them.

In [ ]:
# --- 4.1 Corpus statistics table ---
import pandas as pd

rows = []
for pair in PAIRS:
    for split in raw[pair]:
        rows.append({"pair": pair, "split": split, "n_sentences": raw[pair][split].num_rows})
stats = pd.DataFrame(rows).pivot(index="pair", columns="split", values="n_sentences").fillna(0).astype(int)
print("Sentence counts per pair / split:")
stats

In [ ]:
# --- 4.2 Languages & scripts present ---
LANG_INFO = {
    "en": ("English",  "Latin"),
    "fr": ("French",   "Latin"),
    "es": ("Spanish",  "Latin"),
    "hi": ("Hindi",    "Devanagari"),
    "ta": ("Tamil",    "Tamil"),
}
print("Languages / scripts covered by the four pairs:")
for code_, (name, script) in LANG_INFO.items():
    print(f"  {code_}: {name:8s} — {script} script")

In [ ]:
# --- 4.3 Sentence-length distributions (in characters & words) ---
import numpy as np

def length_sample(pair, side, split="test", k=800):
    ds = raw[pair][split]
    k = min(k, ds.num_rows)
    texts = [ds[i]["translation"][side] for i in range(k)]
    return np.array([len(t) for t in texts]), np.array([len(t.split()) for t in texts])

summary = []
for pair in PAIRS:
    for side in pair.split("-"):
        c, w = length_sample(pair, side)
        summary.append({"pair": pair, "lang": side,
                        "mean_chars": round(c.mean(), 1),
                        "mean_words": round(w.mean(), 1),
                        "p95_words": int(np.percentile(w, 95))})
pd.DataFrame(summary)

In [ ]:
# --- 4.4 Visualisations ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
test_counts = [raw[p]["test"].num_rows for p in PAIRS]
axes[0].bar(PAIRS, test_counts, color="#4C72B0")
axes[0].set_title("Test-split sentence count per pair")
axes[0].set_ylabel("sentences")
for pair in PAIRS:
    _, w = length_sample(pair, "en")
    axes[1].hist(w, bins=30, alpha=0.5, label=pair)
axes[1].set_title("English sentence length (words)")
axes[1].set_xlabel("words per sentence"); axes[1].set_xlim(0, 60)
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# --- 4.5 Concrete examples with non-Latin scripts ---
print("Hindi (Devanagari) example:")
print("   ", raw["en-hi"]["test"][0]["translation"]["hi"])
print("Tamil (Tamil script) example:")
print("   ", raw["en-ta"]["test"][0]["translation"]["ta"])

## 5 · Data Preprocessing
**UTF-8 validation → whitespace normalisation → light cleaning → language mapping → tokenization prep.**
Cleaning is intentionally light: support text is noisy and NLLB is robust.

In [ ]:
# --- 5.1 Language mapping: ISO 639-1  <->  NLLB (FLORES-200) codes ---
LANG_MAP = {
    "en": {"name": "English", "nllb": "eng_Latn"},
    "fr": {"name": "French",  "nllb": "fra_Latn"},
    "es": {"name": "Spanish", "nllb": "spa_Latn"},
    "hi": {"name": "Hindi",   "nllb": "hin_Deva"},
    "ta": {"name": "Tamil",   "nllb": "tam_Taml"},
}
NAME_TO_ISO = {v["name"].lower(): k for k, v in LANG_MAP.items()}
NAME_TO_ISO.update({k: k for k in LANG_MAP})

def to_iso(lang: str) -> str:
    key = lang.strip().lower()
    if key not in NAME_TO_ISO:
        raise ValueError(f"Unsupported language: {lang!r}. Supported: {list(LANG_MAP)}")
    return NAME_TO_ISO[key]

def to_nllb(lang: str) -> str:
    return LANG_MAP[to_iso(lang)]["nllb"]

print("Language mapping ready:")
for k, v in LANG_MAP.items():
    print(f"  {k} -> {v['nllb']}  ({v['name']})")

In [ ]:
# --- 5.2 Cleaning utilities: UTF-8 validation + whitespace normalisation ---
import re, unicodedata

def ensure_utf8(text) -> str:
    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="replace")
    return text.encode("utf-8", errors="replace").decode("utf-8")

def normalize_whitespace(text: str) -> str:
    text = text.replace(" ", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def clean_text(text) -> str:
    text = ensure_utf8(text)
    text = unicodedata.normalize("NFC", text)
    return normalize_whitespace(text)

demo = "  Hello  world!!   \n\n\n  Café  résumé  "
print(repr(demo), "->", repr(clean_text(demo)))

In [ ]:
# --- 5.3 Dataset formatting helper (used later for evaluation batches) ---
def make_eval_records(pair, n=20, split="test"):
    src, tgt = pair.split("-")
    ds = raw[pair][split]
    n = min(n, ds.num_rows)
    return [{
        "src_iso": src, "tgt_iso": tgt,
        "src_text": clean_text(ds[i]["translation"][src]),
        "ref_text": clean_text(ds[i]["translation"][tgt]),
    } for i in range(n)]

print("Formatted record example:\n", make_eval_records("en-fr", n=2)[0])

**Tokenization preparation.** NLLB uses a shared SentencePiece subword tokenizer. We do not tokenise
manually — the model's tokenizer (Section 8) does subword segmentation, adds the source-language token,
and handles padding/truncation. Preparing text = clean it and set the correct source/target codes.

## 6 · Language Detection
We auto-detect the **source** language with **Lingua**, which is accurate on short, noisy text and
returns confidence values so we can reject low-confidence input gracefully.

In [ ]:
# --- 6.1 Build a Lingua detector over our supported languages ---
from lingua import Language, LanguageDetectorBuilder

_SUPPORTED_LINGUA = {
    Language.ENGLISH: "en", Language.FRENCH: "fr", Language.SPANISH: "es",
    Language.HINDI: "hi", Language.TAMIL: "ta",
}
detector = (LanguageDetectorBuilder
            .from_languages(*_SUPPORTED_LINGUA.keys())
            .with_preloaded_language_models().build())
print("Lingua detector built for:", list(_SUPPORTED_LINGUA.values()))

In [ ]:
# --- 6.2 detect_language(): returns iso, NLLB code, confidence; handles unknown ---
CONFIDENCE_THRESHOLD = 0.55

def detect_language(text: str):
    text = clean_text(text)
    if not text:
        return {"iso": None, "nllb": None, "name": None,
                "confidence": 0.0, "is_unknown": True, "reason": "empty input"}
    conf_values = detector.compute_language_confidence_values(text)
    if not conf_values:
        return {"iso": None, "nllb": None, "name": None,
                "confidence": 0.0, "is_unknown": True, "reason": "no language matched"}
    best = conf_values[0]
    iso = _SUPPORTED_LINGUA[best.language]
    confident = best.value >= CONFIDENCE_THRESHOLD
    return {
        "iso": iso if confident else None,
        "nllb": LANG_MAP[iso]["nllb"] if confident else None,
        "name": LANG_MAP[iso]["name"] if confident else None,
        "confidence": round(float(best.value), 3),
        "is_unknown": not confident,
        "reason": None if confident else f"low confidence (<{CONFIDENCE_THRESHOLD})",
        "top_iso": iso,
    }

tests = [
    "My password reset link is not working, please help!",
    "Je n'arrive pas à me connecter à mon compte.",
    "No puedo acceder a mi cuenta, error 500.",
    "मेरा खाता लॉक हो गया है, कृपया मदद करें।",
    "என் கணக்கு பூட்டப்பட்டுள்ளது, உதவவும்.",
    "aksjdhf qwlekjrh zzz",
    "",
]
for t in tests:
    r = detect_language(t)
    tag = r["name"] or f"UNKNOWN ({r['reason']})"
    print(f"[{tag:>22}] conf={r['confidence']:<5} | {t[:45]}")

## 7 · Transformer Architecture Explanation
NLLB is a Transformer **encoder–decoder** (Vaswani et al., 2017).

```
                SOURCE (e.g. Hindi)                 TARGET (e.g. English)
             "मेरा खाता लॉक हो गया"                   "My account is locked"
                     |                                        |
             +---------------+                        +---------------+
             |  Tokenizer +  |                        |  Tokenizer +  |
             |  Pos. Encode  |                        |  Pos. Encode  |
             +-------+-------+                        +-------+-------+
                     |                                        |
        =============v=============            =============v=================
        ||        ENCODER        ||            ||          DECODER          ||
        ||  +-----------------+  ||            ||  | Masked Multi-Head   |  ||
        ||  | Multi-Head      |  ||            ||  | SELF-Attention      |  ||
        ||  | SELF-Attention  |  ||            ||  +---------------------+  ||
        ||  +-----------------+  ||            ||  | CROSS-Attention     |<-++--+
        ||  | Feed-Forward    |  ||            ||  | (Q=decoder,         |  ||  |
        ||  +-----------------+  ||            ||  |  K,V=encoder out) ------+  |
        =============+============            ||  +---------------------+  ||   |
                     |  encoder reps          ||  | Feed-Forward        |  ||   |
                     +------------------------||--+---------------------+  ||   |
                        (Keys & Values)        =============+================   |
                                                            |                   |
                                                    next-token probs -----------+
                                                            |
                                                       DECODED TEXT
```

**Self-Attention** — every token attends to all other tokens in its own sequence, capturing
dependencies like pronoun reference.

**Multi-Head Attention** — attention runs in parallel in several subspaces (syntax, agreement, named
entities), then concatenated — richer than a single head.

**Positional Encoding** — injects word order (no recurrence), distinguishing "agent helped customer"
from "customer helped agent."

**Cross-Attention** — decoder queries with encoder keys/values, so each generated token looks back at
the whole source: the bridge that makes output a *translation*.

**Transfer Learning** — we reuse NLLB's pretraining directly (zero-shot), which is why it works on
low-resource pairs like Tamil.

## 8 · Load the Translation Model

In [ ]:
# --- 8.1 Load NLLB-200 distilled 600M ---
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {n_params/1e6:.1f}M  |  device: {DEVICE}")

In [ ]:
# --- 8.2 Verify our target-language FLORES codes exist in the tokenizer ---
for iso, info in LANG_MAP.items():
    code_ = info["nllb"]
    tok_id = tokenizer.convert_tokens_to_ids(code_)
    ok = tok_id is not None and tok_id != tokenizer.unk_token_id
    print(f"  {code_:9s} -> token id {tok_id}   {'OK' if ok else 'MISSING!'}")
assert all(tokenizer.convert_tokens_to_ids(v['nllb']) != tokenizer.unk_token_id
           for v in LANG_MAP.values()), "A language code is missing!"
print("\nAll target language codes resolve correctly.")

**Language selection in NLLB:** source via `tokenizer.src_lang = "<code>"`; target via
`forced_bos_token_id = tokenizer.convert_tokens_to_ids("<target code>")` (current API — the older
`tokenizer.lang_code_to_id[...]` is deprecated).

## 9 · Translation Pipeline
`customer message + target language` → detect source → protect identifiers → tokenize → translate →
decode → restore identifiers → return. Protection utilities are defined here and explained in Section 10.

In [ ]:
# --- 9.1 Identifier-protection utilities (detailed in Section 10) ---
import re

PRODUCT_NAMES = ["Lexora AI", "Lexora", "AcmeCloud", "PayPro", "QuickShip", "InvoiceX"]

_PATTERNS = [
    ("url",     re.compile(r"https?://\S+|www\.\S+")),
    ("email",   re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")),
    ("errcode", re.compile(r"\b(?:0x[0-9A-Fa-f]+|(?:ERR|ERROR|ERRNO|CODE|E|HTTP)[-_ ]?\d{2,}|\d{3}\s?error)\b", re.I)),
    ("ticket",  re.compile(r"#\w[\w-]*")),
    ("version", re.compile(r"\bv?\d+\.\d+(?:\.\d+)*\b")),
    ("product", re.compile(r"\b(?:" + "|".join(re.escape(p) for p in sorted(PRODUCT_NAMES, key=len, reverse=True)) + r")\b")),
]
_EMOJI = re.compile("[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF"
                    "\U0001F1E6-\U0001F1FF" "\U0001F900-\U0001F9FF" "\U0000FE00-\U0000FE0F" "]+")

def _placeholder(i): return f"PHOLDER{i}X"

def protect(text: str):
    mapping = {}; idx = 0
    def _sub(pattern):
        nonlocal idx, text
        def repl(m):
            nonlocal idx
            ph = _placeholder(idx); mapping[ph] = m.group(0); idx += 1; return ph
        text = pattern.sub(repl, text)
    for _n, pat in _PATTERNS: _sub(pat)
    _sub(_EMOJI)
    return text, mapping

def restore(text: str, mapping: dict) -> str:
    for ph, original in mapping.items():
        num = re.search(r"\d+", ph).group(0)
        pat = re.compile(r"P\s*H\s*O\s*L\s*D\s*E\s*R\s*" + num + r"\s*X", re.I)
        text = pat.sub(lambda _m: original, text); text = text.replace(ph, original)
    return text

_m, _map = protect("Login to https://acme.io fails with ERR-500 on Lexora v2.1.3 🎉 mail me@x.com")
print("masked :", _m); print("mapping:", _map); print("restore:", restore(_m, _map))

In [ ]:
# --- 9.2 The core translation function ---
@torch.inference_mode()
def translate_message(text: str, target_language: str, source_language: str = None,
                      num_beams: int = 4, max_new_tokens: int = 256):
    stages = {"input": text}
    cleaned = clean_text(text); stages["cleaned"] = cleaned

    if source_language:
        src_iso = to_iso(source_language)
        det = {"iso": src_iso, "nllb": LANG_MAP[src_iso]["nllb"], "name": LANG_MAP[src_iso]["name"],
               "confidence": 1.0, "is_unknown": False, "reason": "manual override"}
    else:
        det = detect_language(cleaned)
    stages["detection"] = det
    if det["is_unknown"]:
        return {"translation": None, "error": "Could not confidently detect source language.",
                "detection": det, "stages": stages}

    tgt_iso = to_iso(target_language)
    src_code, tgt_code = det["nllb"], LANG_MAP[tgt_iso]["nllb"]
    stages["src_code"], stages["tgt_code"] = src_code, tgt_code
    if det["iso"] == tgt_iso:
        return {"translation": cleaned, "detection": det, "note": "source == target",
                "stages": {**stages, "final": cleaned}}

    masked, mapping = protect(cleaned)
    stages["masked"] = masked; stages["protected_map"] = mapping

    tokenizer.src_lang = src_code
    inputs = tokenizer(masked, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    stages["input_token_ids"] = inputs["input_ids"][0].tolist()
    stages["input_tokens"] = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    forced_bos = tokenizer.convert_tokens_to_ids(tgt_code)
    generated = model.generate(**inputs, forced_bos_token_id=forced_bos,
                               num_beams=num_beams, max_new_tokens=max_new_tokens)
    stages["output_token_ids"] = generated[0].tolist()
    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    stages["decoded"] = decoded
    final = normalize_whitespace(restore(decoded, mapping)); stages["final"] = final

    return {"translation": final, "detection": det, "source_language": det["name"],
            "target_language": LANG_MAP[tgt_iso]["name"], "stages": stages}

In [ ]:
# --- 9.3 Smoke-test across all four required pairs (both directions) ---
smoke = [
    ("I forgot my password and cannot log in.", "fr"),
    ("Je voudrais un remboursement pour ma commande.", "en"),
    ("Mi pago fue rechazado dos veces.", "en"),
    ("Please translate this to Spanish: my order is late.", "es"),
    ("My account is locked after error 429.", "hi"),
    ("नमस्ते, मेरा भुगतान विफल हो गया।", "en"),
    ("I need help resetting my PIN.", "ta"),
    ("எனது கணக்கு முடக்கப்பட்டது.", "en"),
]
for text, tgt in smoke:
    out = translate_message(text, tgt)
    print(f"[{out['detection']['name']:>7} -> {LANG_MAP[to_iso(tgt)]['name']:<7}] "
          f"{text}\n    => {out['translation']}\n")

## 10 · Post-Processing — preserving technical accuracy
Strategy = **mask → translate → restore**: protected spans (error codes, product names, versions,
emails, URLs, emoji) become placeholders NLLB copies through, then are restored after decoding.

In [ ]:
# --- 10.1 Demonstrate identifier & emoji preservation ---
cases = [
    ("Order via QuickShip failed with ERR-500 at https://acme.io/track — help! 😤", "fr"),
    ("Update Lexora to v2.1.3 and email support@lexora.ai if 0x80070005 persists.", "es"),
    ("Payment on PayPro shows HTTP 402. My ticket is #INV-99823. Thanks! 🙏", "hi"),
]
for text, tgt in cases:
    out = translate_message(text, tgt)
    print("IN :", text); print("OUT:", out["translation"])
    kept = [v for v in out["stages"]["protected_map"].values() if v in out["translation"]]
    print("Preserved verbatim:", kept, "\n")

In [ ]:
# --- 10.2 Verify nothing protected was lost ---
def preservation_report(text, tgt):
    out = translate_message(text, tgt)
    originals = list(out["stages"].get("protected_map", {}).values())
    kept = [o for o in originals if o in (out["translation"] or "")]
    lost = [o for o in originals if o not in (out["translation"] or "")]
    return {"n_protected": len(originals), "kept": kept, "lost": lost}

for text, tgt in cases:
    print(tgt, "->", preservation_report(text, tgt))

## 11 · Evaluation — BLEU · chrF · COMET
Held-out OPUS-100 references, three complementary metrics. chrF and COMET matter most for Hindi/Tamil.

In [ ]:
# --- 11.1 Generate translations for a held-out sample of each pair ---
import sacrebleu
EVAL_N = 25
set_seed()

def evaluate_pair(pair, n=EVAL_N):
    src, tgt = pair.split("-")
    recs = make_eval_records(pair, n=n)
    sources = [r["src_text"] for r in recs]
    refs = [r["ref_text"] for r in recs]
    hyps = [translate_message(s, target_language=tgt, source_language=src)["translation"] or "" for s in sources]
    return {"pair": f"{src}->{tgt}", "n": len(hyps),
            "BLEU": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
            "chrF": round(sacrebleu.corpus_chrf(hyps, [refs]).score, 2),
            "sources": sources, "hyps": hyps, "refs": refs}

results = [evaluate_pair(p) for p in PAIRS]
import pandas as pd
score_df = pd.DataFrame([{k: r[k] for k in ("pair", "n", "BLEU", "chrF")} for r in results])
print("BLEU / chrF on held-out OPUS-100 test sentences:")
score_df

In [ ]:
# --- 11.2 COMET (neural metric). Heavy (~2GB); guarded so the notebook still completes. ---
comet_scores = {}
try:
    from comet import download_model, load_from_checkpoint
    comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
    for r in results:
        data = [{"src": s, "mt": h, "ref": ref} for s, h, ref in zip(r["sources"], r["hyps"], r["refs"])]
        pred = comet_model.predict(data, batch_size=8, gpus=1 if DEVICE == "cuda" else 0, progress_bar=False)
        comet_scores[r["pair"]] = round(float(pred["system_score"]), 4)
    print("COMET system scores:", comet_scores)
except Exception as e:
    print("COMET step skipped (environment/download issue):", repr(e))
    print("BLEU and chrF above remain valid; re-run this cell to retry COMET.")

In [ ]:
# --- 11.3 Consolidated score table + qualitative examples ---
final_scores = score_df.copy()
final_scores["COMET"] = final_scores["pair"].map(comet_scores).fillna("n/a")
print("=== Final quality scores ===")
print(final_scores.to_string(index=False))
print("\n=== Qualitative sample (en->fr) ===")
r0 = results[0]
for i in range(min(3, len(r0["hyps"]))):
    print("SRC:", r0["sources"][i]); print("HYP:", r0["hyps"][i]); print("REF:", r0["refs"][i]); print("-"*60)

**Reading the results.** Higher is better. French/Spanish score highest; Hindi/Tamil are relatively
stronger on chrF/COMET than word-BLEU — which is why we report all three. NLLB is zero-shot here, so
absolute BLEU looks modest even when outputs are fluent; COMET gives a fairer picture. Raise `EVAL_N`
for a tighter estimate.

## 12 · Interactive Demo
Realistic support scenarios covering password reset, account locked, payment failure, shipping delay,
refund, error codes, mixed-language, emoji, typos, and abbreviations.

In [ ]:
# --- 12.1 Pretty demo helper ---
def demo(text, target_language, source_language=None):
    out = translate_message(text, target_language, source_language)
    print("-" * 72)
    if out.get("translation") is None:
        print("INPUT :", text); print("ERROR :", out.get("error")); return
    d = out["detection"]
    print(f"INPUT  ({d['name']}, conf={d['confidence']}): {text}")
    print(f"OUTPUT ({out['target_language']}): {out['translation']}")

scenarios = [
    ("Password reset",  "I clicked 'forgot password' but never got the reset email.", "fr"),
    ("Account locked",  "My account got locked after too many login attempts.", "es"),
    ("Payment failure", "My payment on PayPro keeps failing with ERR-402.", "hi"),
    ("Shipping delay",  "My QuickShip order #INV-99823 is 5 days late, where is it?", "ta"),
    ("Refund request",  "I want a full refund for the duplicate charge on my card.", "fr"),
    ("Error code",      "The app crashes on startup with code 0x80070005.", "es"),
    ("Mixed language",  "Hola, my compte is blocked, por favor help me reset it.", "en"),
    ("Emoji",           "Still waiting for my refund 😡😡 this is unacceptable!", "hi"),
    ("Typos",           "i cant loginn to my acount plz help urgnt", "fr"),
    ("Abbreviations",   "FYI the pmt failed & I need a refund ASAP, thx.", "es"),
]
for label, text, tgt in scenarios:
    print(f"\n### {label}"); demo(text, tgt)

In [ ]:
# --- 12.2 Try your own message (uncomment to run interactively) ---
# msg = input("Enter a support message: ")
# tgt = input("Target language (English/French/Spanish/Hindi/Tamil): ")
# demo(msg, tgt)
demo("Bonjour, je n'ai toujours pas reçu mon remboursement pour la commande #INV-1234.", "English")

## 13 · Dry Run — every stage of the pipeline

In [ ]:
# --- 13.1 Full-stage trace for one representative message ---
example = "Hi, my QuickShip order #INV-9932 is late & I got ERR-500 at https://acme.io 😤 plz help!"
target = "Hindi"
out = translate_message(example, target)
S = out["stages"]

def header(t): print("\n" + "=" * 72 + f"\n{t}\n" + "=" * 72)

header("STAGE 1 — RAW INPUT"); print(S["input"])
header("STAGE 2 — CLEANING (UTF-8, NFC, whitespace)"); print(S["cleaned"])
header("STAGE 3 — LANGUAGE DETECTION")
for k, v in S["detection"].items(): print(f"  {k:12s}: {v}")
header("STAGE 4 — IDENTIFIER MASKING")
print("masked text :", S["masked"]); print("placeholders:", S["protected_map"])
header("STAGE 5 — TOKENIZATION (source lang = %s)" % S["src_code"])
print("first 20 subword tokens:", S["input_tokens"][:20])
print("first 20 token ids     :", S["input_token_ids"][:20])
print("total input tokens     :", len(S["input_token_ids"]))
header("STAGE 6 — MODEL GENERATION (target forced to %s)" % S["tgt_code"])
print("output token ids (first 20):", S["output_token_ids"][:20])
print("total output tokens        :", len(S["output_token_ids"]))
header("STAGE 7 — DECODING (subwords -> text)"); print(S["decoded"])
header("STAGE 8 — POST-PROCESSING (restore identifiers)"); print(S["final"])
header("FINAL TRANSLATION  (%s -> %s)" % (out["source_language"], out["target_language"]))
print(out["translation"])

---
### Summary & disclosure
- **Architecture:** Transformer encoder–decoder (`facebook/nllb-200-distilled-600M`) used consistently.
- **Pipeline:** clean → detect (Lingua) → protect identifiers → tokenize (SentencePiece) → translate
  (forced target) → decode → restore identifiers.
- **Language pairs:** English ↔ French / Spanish / Hindi / Tamil (both directions).
- **Robustness:** handles emoji, typos, abbreviations, mixed-language; preserves error codes, product
  names, versions, emails, URLs.
- **Evaluation:** BLEU + chrF (sacrebleu) and COMET on held-out OPUS-100 references.
- **Disclosed tools:** Hugging Face Transformers, NLLB-200, Lingua, sacrebleu, unbabel-comet, OPUS-100.
  No fine-tuning required (zero-shot transfer learning).